Sistema completo

In [ ]:
k = 0
n = Q.shape[0]
#print("n= ",n, "Q.shape[0]")
m = A.shape[0]
#print("m= ",m, "A.shape[0]")
p = F.shape[0]
#print("p= ",p, "F.shape[0]")

tol = 1e-6
kmax = 100

#print("El rango de A es", np.linalg.matrix_rank(A))

AT = A.T
FT = F.T

# Initial values
x = np.ones(n)
lamda = np.zeros(m)
mu = np.ones(p)
z = np.ones(p)

sigma = 0.5 # Valor fijo
tau = sigma * np.dot(mu, z) / p

# Dataframes that store mu and z values on every iteration.
# Each iteration is a new row added after this.
# Used for the graphs
mudf = pd.DataFrame(columns=[f"{i}" for i in range(len(mu))])
zdf = pd.DataFrame(columns=[f"{i}" for i in range(len(z))])
mudf.loc[k] = mu
zdf.loc[k] = z

v1 = Q @ x + AT @ lamda - FT @ mu + c
v2 = A @ x - b
v3 = -F @ x + d + z
v4 = np.multiply(mu, z)  # Element-wise product
ld = np.concatenate((v1, v2, v3, v4), 0)
norma_cnpo = np.linalg.norm(ld,np.inf)
w = np.zeros((p, 1))

# Initialize an empty DataFrame to store the iteration results
highlighted_df = pd.DataFrame(columns=range(p))  # drop/test inequalities → p columns

while norma_cnpo > tol and k < kmax:
    # Update diagonal matrices Z and U inside the loop
    # Initial residuals
    Z = np.diag(z)
    U = np.diag(mu)
    v4_pert = np.multiply(mu, z) - tau * np.ones(p)

    ### KKT Matrix
    row1 = np.hstack((Q, AT, -FT, np.zeros((n, p))))
    row2 = np.hstack((A, np.zeros((m, m + p + p))))
    row3 = np.hstack((-F, np.zeros((p, m + p)), np.identity(p)))
    row4 = np.hstack((np.zeros((p, n + m)), Z, U))

    K = np.vstack((row1, row2, row3, row4))
    
    #D = np.diag(mu / z)
    #G = Q+FT@D@F
    #for i in range(p):
    #    w[i] = F[i, :] @ x - d[i] - (tau / mu[i])
    #w = w.ravel()
        
    #dg = Q @ x + AT @ lamda - FT@mu + c + FT@D@w
    
    # Define K as a block matrix
    #m = A.shape[0]
    #K = np.block([
    #    [G, AT],
    #    [A, np.zeros((m, m))]
    #])
    
    # Calculate the reciprocal condition number of G
    condK = np.linalg.cond(K,1)
    ld_pert = np.concatenate((v1, v2, v3, v4_pert), 0)
    
    # Define ld
    #ld = -np.concatenate([dg, A @ x - b])
    norma_cnpo = np.linalg.norm(ld_pert, np.inf)
    
    # Solve the linear system and catch errors
    
    w_vector = scipy.linalg.solve(K, -ld_pert) # Reduced system
    
    # Update the sections of the w_vector
    wx = w_vector[0:n]
    wlamda = w_vector[n:n + m]
    wmu = w_vector[n + m:n + m + p]
    wz = w_vector[n + m + p:]
    
    ### Step size
    alpha_mu = paso_intpoint(mu, wmu)
    alpha_z  = paso_intpoint(z, wz)
    #alpha    = 0.995 * min(alpha_mu, alpha_z)
    alpha    = min(alpha_mu, alpha_z)
    #print("alpha= " , alpha)
    
    # remember something
    perc_mu = wmu/mu
    perc_z  = wz/z
    
    # Update variables
    x += alpha * wx
    mu += alpha * wmu
    lamda += alpha * wlamda
    z += alpha * wz

    mudf.loc[k] = mu
    zdf.loc[k] = z
    
    # Update tau and residuals
    tau = sigma * np.dot(mu, z) / (p)
    k += 1
    
    # Recalculate residuals
    v1 = Q @ x + AT @ lamda - FT @ mu + c
    v2 = A @ x - b
    v3 = -F @ x + d + z
    v4 = np.multiply(mu, z)  # Element-wise product
    
    ld1 = np.concatenate((v1, v2, v3, v4), 0)
    norma_cnpo = np.linalg.norm(ld1,np.inf)
    
    #print("\niter=", k, "\t", "||cnpo||=", norma_cnpo)
    #print("Condition number of G:", np.linalg.cond(G,1))
    #print("rcond(G)", (1/np.linalg.cond(G,1)))
    #print("tau =",tau)
    #print(z)
    #print(mu)
    
    mask = mu*z <= 1e-5
    
    #print('cuantos chicos mu*z = %g, vector\n' % (sum(mask)), (mu*z)[mask])
    norm_before = norma_cnpo 

    red_mu = []
    if all(mask):
        pr_before   = np.linalg.norm(v2, np.inf)
        inq_before  = np.linalg.norm(v3, np.inf)
        cmp_before  = np.max(v4)
        tau_before  = tau
        cond_before = condK
        obj_before  = 0.5 * x @ Q @ x + c @ x
        norm_before = norma_cnpo

        #neg_mu_mask = (-0.52 < perc_mu) & (perc_mu < -0.48)
        #const_z_mask = (-0.01 < perc_z) & (perc_z < 0.01)
        grow_z_mask = (perc_z > -0.03) #& (perc_z < 0.01)
        neg_mu = np.arange( len(mask) )[grow_z_mask]
        
        highlighted_df = highlight_df(mu,perc_mu,z,perc_z,Q,k)
        
        #print('mus chicos: vector\n', mu[neg_mu])
        if set(red_mu).issubset( neg_mu ):
            print ('IS subset: GOOD')
        else:
            print ('FAILS subset condition: BAD')
            
        #print('mus chicos: vector\n', neg_mu)
        #print('  change in percentages for mu \n', perc_mu[neg_mu] )
        #print('zs tending to positive contants\n', z[neg_mu] )
        #print('  Largest and smallest change for percentages in entries of z  \n', min(perc_z[neg_mu]), max(perc_z[neg_mu] ))
        red_mu = neg_mu.copy()
    
    # So here is when we start getting close to the issues of the matrix
    if norma_cnpo <= tol or k == kmax:
        print(f"HELLO K={k}")
        k_red = 0
        tol_red = 1e-10
        max_red = 5

        while k_red < max_red:
        #while np.linalg.norm(ld1, np.inf) > tol_red and k_red < max_red:
            Z = np.diag(z)
            U = np.diag(mu)
            highlighted_columns, regressed_columns = display_highlights(highlighted_df)
            M1, U1, ld1 = build_reduced_system(Q, AT, FT, U, A, F, Z, mu, x, lamda, c, b, d, tau, highlighted_columns)

            print("cond(M1, 1-norm):", np.linalg.cond(M1, 1))

            # SOLVE THE NEW 3X3 SYSTEM
            w_vector1 = scipy.linalg.solve(M1, ld1)

            ### OK SO UP TILL NOW I JUST SOLVED ONE ITERATION OF THIS METHOD AND NOW I NEED TO OBSERVE THESE RESULTS
            ## NOW I NEED TO CHANGE THESE

            wx1     = w_vector1[:n] # should be the same ?
            wlamda1 = w_vector1[n:n + m] # should be the same ?
            Dwmu1    = w_vector1[n + m:]
            
            wmu1 = U1 @ Dwmu1  # Divide each element of wmu1 by the corresponding diagonal element of U

            # OK so now what? We need to analyse the problem and see if we're getting any closer.
            # Here's the vector transformed back into the original size problem
            full_wmu = np.zeros_like(mu)
            active_indices = [i for i in range(p) if i not in highlighted_columns]
            for i, idx in enumerate(active_indices):
                full_wmu[idx] = wmu1[i]

            # ────────── complete here ──────────
            # 1. Reconstruct Δz from the 3rd KKT row (primal feasibility)
            # δz = F δx - (Fx - d - z)  == F @ wx1 - v3  (since v3 = -F@x + d + z)
            full_wz = F @ wx1 - (F @ x - d - z)

            # Recompute step sizes for the reduced-step directions **DOUBLE CHECK THIS LATER
            alpha_mu = paso_intpoint(mu, full_wmu)   # ensures μ + α·Δμ ≥ (1-τ)μ > 0
            alpha_z  = paso_intpoint(z,  full_wz)    # ensures z + α·Δz ≥ (1-τ)z > 0
            alpha    = 0.995 * min(alpha_mu, alpha_z)

            # 2. Apply the step with the same step‑size alpha
            x      += alpha * wx1
            lamda  += alpha * wlamda1
            mu     += alpha * full_wmu
            z      += alpha * full_wz

            #print("max change in x:", np.max(np.abs(alpha*wx1)))
            #print("max change in mu:", np.max(np.abs(alpha*full_wmu)))
            #print("max change in z:", np.max(np.abs(alpha*full_wz)))

            # 3. Recompute residuals for diagnostics
            v1 = Q @ x + AT @ lamda - FT @ mu + c      # dual residual
            v2 = A @ x - b                             # primal residual
            v3 = -F @ x + d + z                        # inequality residual
            v4 = mu * z                                # complementarity product
            ld1 = np.concatenate((v1, v2, v3, v4))
            norm_after = np.linalg.norm(ld1, np.inf)

            # 4. Refresh τ and bump iteration counter (optional, for bookkeeping)
            tau = np.dot(mu, z) / (2 * p)
            k  += 1
            k_red += 1

            # Optional: save to Excel
            
        summary_df, pr_after, inq_after, cmp_after, tau_after, cond_after, obj_after, norm_after = \
                progress_summary_df_clean(v2, v3, v4, tau, G, x, Q, c, ld1,
                                        norm_before, pr_before, inq_before, cmp_before,
                                        tau_before, cond_before, obj_before)

        display(summary_df)
        summary_df.to_excel(f"progress_summary_{mat_files}.xlsx", index=False)

        print("Condition number (1-norm):", cond_after)
        print("Objective after:", obj_after)
        print(f"k={k}")
        print(f"k_red={k_red}")